In [1]:
import common as cm
import pandas as pd
import os
import numpy as np

In [2]:
# Load near-discrete PRS models once (mode share > 0.5: more than half of individuals share one score)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_eur.csv"))
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "prs_phenocode"], qc_df.loc[qc_df["flag_mode_0.5"], "model_file"]))
records = []

for icd, phenocode in zip(cm.phecode2icd["icd"], cm.phecode2icd["phenocode"]):
    train_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training", f"{icd}.csv")
    test_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_testing", f"{icd}.csv")

    # Skip traits missing either ranking file
    if not (os.path.exists(train_path) and os.path.exists(test_path)):
        continue

    # Read ranking files
    # Columns: auc, prs_phenocode, model_file, n_samples, n_cases
    train_rank = pd.read_csv(train_path)
    test_rank = pd.read_csv(test_path)

    # Keep only the target trait's own PRS models (within-trait: up to 3 methods)
    train_own = train_rank[train_rank["prs_phenocode"] == phenocode].copy()
    test_own = test_rank[test_rank["prs_phenocode"] == phenocode].copy()

    # Skip traits without their own PRS in either half
    if train_own.empty or test_own.empty:
        continue

    # Flag near-discrete models in both halves (QC is defined on the full EUR score distribution)
    is_flagged_train = pd.Series([(p, m) in flagged for p, m in zip(train_rank["prs_phenocode"], train_rank["model_file"])], index=train_rank.index)
    is_flagged_test = pd.Series([(p, m) in flagged for p, m in zip(test_rank["prs_phenocode"], test_rank["model_file"])], index=test_rank.index)
    train_rank.loc[is_flagged_train, "auc"] = 0.5
    test_rank.loc[is_flagged_test, "auc"] = 0.5

    # Select the best within-trait PRS method on the training half
    train_best = train_own.loc[train_own["auc"].idxmax()]
    best_model = train_best["model_file"]

    # Locate the same PRS model in the testing half
    test_match = test_own.loc[test_own["model_file"] == best_model]
    if test_match.empty:
        continue
    test_row = test_match.iloc[0]

    # Oracle best within-trait PRS: selected directly on the testing half
    oracle_row = test_own.loc[test_own["auc"].idxmax()]

    # Map the selected PRS's phenocode back to its ICD code (equals the target ICD here)
    prs_phecode = str(train_best["prs_phenocode"])
    prs_icd = cm.phecode2icd.get(prs_phecode, np.nan)

    records.append({
        "icd": icd,
        "auc": test_row["auc"],                    # train-selected method, evaluated on testing half
        "auc_train": train_best["auc"],            # same model, AUC on training half
        "model_file": best_model,                  # selected method (C+T / LDpred2 / lassosum2)
        "prs_phecode": prs_phecode,
        "prs_icd": prs_icd,
        "auc_oracle": oracle_row["auc"],           # oracle best within-trait AUC on testing half
        "n_case": test_row["n_cases"],
        "n_sample": test_row["n_samples"]
    })

# Assemble and save the best within-trait single summary table
df = pd.DataFrame(records)
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "aou_besttargetsingle_result.csv")
df.to_csv(report_file, index=False)

In [4]:
# Load near-discrete PRS models once (mode share > 0.5: more than half of individuals share one score)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_eur.csv"))
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "prs_phenocode"], qc_df.loc[qc_df["flag_mode_0.5"], "model_file"]))
records = []

for icd, phenocode in zip(cm.phecode2icd["icd"], cm.phecode2icd["phenocode"]):
    train_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training", f"{icd}.csv")
    ukb_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "UKB", f"{icd}.csv")

    # Skip traits missing either ranking file
    if not (os.path.exists(train_path) and os.path.exists(ukb_path)):
        continue

    # Read ranking files
    # Columns: auc, prs_phenocode, model_file, n_samples, n_cases
    train_rank = pd.read_csv(train_path)
    ukb_rank = pd.read_csv(ukb_path)

    # Keep only the target trait's own PRS models (within-trait: up to 3 methods)
    train_own = train_rank[train_rank["prs_phenocode"] == phenocode].copy()
    ukb_own = ukb_rank[ukb_rank["prs_phenocode"] == phenocode].copy()

    # Skip traits without their own PRS in either cohort
    if train_own.empty or ukb_own.empty:
        continue

    # Set AUC of single-SNP C+T scores to 0.5 in both cohorts
    is_flagged_train = pd.Series([(p, m) in flagged for p, m in zip(train_rank["prs_phenocode"], train_rank["model_file"])], index=train_rank.index)
    is_flagged_test = pd.Series([(p, m) in flagged for p, m in zip(test_rank["prs_phenocode"], test_rank["model_file"])], index=test_rank.index)
    train_rank.loc[is_flagged_train, "auc"] = 0.5
    test_rank.loc[is_flagged_test, "auc"] = 0.5

    # Select the best within-trait PRS method on the AoU training half
    train_best = train_own.loc[train_own["auc"].idxmax()]
    best_model = train_best["model_file"]

    # Locate the same PRS model in UKB
    ukb_match = ukb_own.loc[ukb_own["model_file"] == best_model]
    if ukb_match.empty:
        continue
    ukb_row = ukb_match.iloc[0]

    # Oracle best within-trait PRS: selected directly in UKB
    oracle_row = ukb_own.loc[ukb_own["auc"].idxmax()]

    # Map the selected PRS's phenocode back to its ICD code (equals the target ICD here)
    prs_phecode = str(train_best["prs_phenocode"])
    prs_icd = cm.phecode2icd.get(prs_phecode, np.nan)

    records.append({
        "icd": icd,
        "auc": ukb_row["auc"],                     # AoU-train-selected method, evaluated in UKB
        "auc_train": train_best["auc"],            # same model, AUC on AoU training half
        "model_file": best_model,                  # selected method (C+T / LDpred2 / lassosum2)
        "prs_phecode": prs_phecode,
        "prs_icd": prs_icd,
        "auc_oracle": oracle_row["auc"],           # oracle best within-trait AUC in UKB
        "n_case": ukb_row["n_cases"],
        "n_sample": ukb_row["n_samples"]
    })

# Assemble and save the best within-trait single summary table
df = pd.DataFrame(records)
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "ukb_besttargetsingle_result.csv")
df.to_csv(report_file, index=False)